# # Phases 2–5 — Evaluation Protocols, Baselines, ML Benchmark, Statistics
# Rock Fragmentation Prediction from Blast Parameters
#
# This notebook fits **empirical baselines** (Kuznetsov equation values as
# published, multivariate power-law regression) and **ML models** (Ridge,
# Random Forest, Extra Trees, XGBoost, LightGBM, CatBoost, SVR, Gaussian
# Process, ANN) under three leakage-aware evaluation protocols:
#
# - **P1 — paper split**: 97 development blasts / 13 original validation
#   blasts (reproduces the split used by Hudaverdi et al. 2011 and
#   Kulatilake et al. 2010, for direct literature comparison).
# - **P2 — repeated grouped K-fold CV**: 10 repeats × 5 folds over all 110
#   blasts, with blasts that share *identical* inputs forced into the same
#   fold (no duplicate-row leakage). This is the **main result**.
# - **P3 — leave-one-site-out (LOSO)**: each of the 10 mines held out in
#   turn — tests generalisation to a brand-new quarry.
#
# All hyper-parameters are tuned with Optuna **inside** the training fold of
# every outer split (nested CV) — the test fold is touched exactly once.
#
# **Kaggle setup:** Add Input → your `blast_master_110.csv` dataset.
# Internet ON (Optuna/shap are pre-installed on Kaggle; the `!pip install`
# line below is a no-op fallback). Expect ~30–60 min runtime for the full
# settings (`N_TRIALS=40`, `n_repeats=10`) on a Kaggle CPU instance.

In [ ]:
# !pip install -q optuna shap  # already present on Kaggle; harmless if run again
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IN_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("../data/processed")
CSV = next(IN_ROOT.rglob("blast_master_110.csv"))
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
for sub in ["figures", "tables", "models"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV)
FEATURES = ["S_B", "H_B", "B_D", "T_B", "Pf", "XB", "E"]
TARGET = "X50"
X, y = df[FEATURES].values, df[TARGET].values
SITE = df["site"].values
print(CSV, df.shape)

for m in ["sklearn", "xgboost", "lightgbm", "catboost", "shap", "optuna"]:
    try:
        print(m, __import__(m).__version__)
    except ImportError:
        print(m, "NOT INSTALLED — run the pip install cell above")

# ## 2.1 Metrics and CV splitters (Phase 2 — evaluation design)
# `DUP_GROUP` assigns every duplicate-input cluster (see Notebook 01 §1.7) a
# single group id, so `repeated_group_kfold` never splits a duplicate pair
# across train/test.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "VAF": (1 - np.var(y_true - y_pred) / np.var(y_true)) * 100,
        "a20": np.mean(np.abs(y_pred / y_true - 1) <= 0.20),
    }


DUP_GROUP = df.groupby(FEATURES, sort=False).ngroup().values


def repeated_group_kfold(groups, n_splits=5, n_repeats=10, seed=SEED):
    """Yields (rep, fold, train_idx, test_idx). Duplicate-input blasts always
    land in the same fold together."""
    rng = np.random.RandomState(seed)
    uniq = np.unique(groups)
    for r in range(n_repeats):
        perm = rng.permutation(uniq)
        for k, fold_groups in enumerate(np.array_split(perm, n_splits)):
            test = np.isin(groups, fold_groups)
            yield r, k, np.where(~test)[0], np.where(test)[0]


def loso_splits(sites):
    """rep is always 0 (one pass over sites); fold = held-out site name."""
    for s in np.unique(sites):
        test = sites == s
        yield 0, s, np.where(~test)[0], np.where(test)[0]


def paper_split():
    test = (df["paper_split"] == "validation").values
    yield "paper", 0, np.where(~test)[0], np.where(test)[0]


print(f"{len(np.unique(DUP_GROUP))} independent groups from {len(df)} blasts "
      f"({len(df) - len(np.unique(DUP_GROUP))} duplicate rows folded in)")

# ## 2.2 Baseline B1 — Kuznetsov / Kuz-Ram (published values)
# The classic Kuznetsov (1973) / Cunningham Kuz–Ram equation needs
# **absolute** burden, spacing, bench height, charge mass and an empirical
# rock factor `A` — quantities not present in the ratio-based database.
# We therefore use the Kuznetsov predictions **already published** by
# Kulatilake et al. (2010, Tables 4–5) for the 12 blasts they cover, and
# compare them against our own out-of-fold predictions on the same blasts
# in §2.6. This is standard practice in this literature (every paper that
# uses this database does the same, e.g. Amoako et al. 2022).

In [ ]:
KUZ = {"En13": .44, "Ru7": .65, "Mg8": .39, "Mg9": .30, "Rc1": .72, "Mr12": .24,
       "Db10": .08, "Sm8": .35, "Oz8": .20, "Oz9": .23, "Ad23": .11, "Ad24": .13}
MVR_PUB = {"En13": .39, "Ru7": .51, "Mg8": .40, "Mg9": .24, "Rc1": .52, "Mr12": .16,
           "Db10": .16, "Sm8": .19, "Oz8": .17, "Oz9": .16, "Ad23": .18, "Ad24": .14}
ANN_PUB = {"En13": .42, "Ru7": .63, "Mg8": .37, "Mg9": .28, "Rc1": .43, "Mr12": .18,
           "Db10": .33, "Sm8": .19, "Oz8": .14, "Oz9": .19, "Ad23": .21, "Ad24": .22}
k = df[df.blast_id.isin(KUZ)]
pd.DataFrame({
    "Kuznetsov (Kulatilake 2010)": metrics(k.X50, k.blast_id.map(KUZ)),
    "MVR (Kulatilake 2010)": metrics(k.X50, k.blast_id.map(MVR_PUB)),
    "ANN (Kulatilake 2010)": metrics(k.X50, k.blast_id.map(ANN_PUB)),
}).T.round(3)

# ## 2.3 Model zoo (Phase 3–4): B2/B3 baselines + ML models
# `MVR` reproduces the Hudaverdi/Kulatilake power-law form
# `ln(X50) = ln(a) + Σ b_i ln(x_i)` as ordinary least squares on log-inputs.
# Tree/SVR/GPR/ANN are wrapped so the target can optionally be modelled in
# log-space (`TransformedTargetRegressor`) while predictions are returned on
# the original (metres) scale.

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor


def logy(est):
    return TransformedTargetRegressor(regressor=est, func=np.log, inverse_func=np.exp)


def build(name, p):
    if name == "Mean":
        return DummyRegressor()
    if name == "MVR":
        return logy(make_pipeline(FunctionTransformer(np.log), LinearRegression()))
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=p["alpha"]))
    if name == "RF":
        return RandomForestRegressor(random_state=SEED, n_jobs=-1, **p)
    if name == "ET":
        return ExtraTreesRegressor(random_state=SEED, n_jobs=-1, **p)
    if name == "XGB":
        return XGBRegressor(random_state=SEED, n_jobs=4, verbosity=0, **p)
    if name == "LGBM":
        return LGBMRegressor(random_state=SEED, verbose=-1, **p)
    if name == "CatB":
        return CatBoostRegressor(random_seed=SEED, verbose=0, **p)
    if name == "SVR":
        return logy(make_pipeline(StandardScaler(), SVR(**p)))
    if name == "GPR":
        kern = ConstantKernel() * RBF(length_scale=np.ones(len(FEATURES))) + WhiteKernel()
        return logy(make_pipeline(StandardScaler(), GaussianProcessRegressor(
            kernel=kern, normalize_y=True, n_restarts_optimizer=5, random_state=SEED)))
    if name == "ANN":
        layers = tuple(p[f"n{i}"] for i in range(p["n_layers"]))
        return logy(make_pipeline(StandardScaler(), MLPRegressor(
            hidden_layer_sizes=layers, activation=p["act"], alpha=p["alpha"],
            solver="lbfgs", max_iter=5000, random_state=SEED)))
    raise ValueError(name)


def space(name, t):
    if name == "Ridge":
        return {"alpha": t.suggest_float("alpha", 1e-3, 100, log=True)}
    if name in ("RF", "ET"):
        return {"n_estimators": t.suggest_int("n_estimators", 200, 800, step=100),
                "max_depth": t.suggest_int("max_depth", 2, 12),
                "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 8),
                "max_features": t.suggest_float("max_features", 0.3, 1.0)}
    if name == "XGB":
        return {"n_estimators": t.suggest_int("n_estimators", 100, 1000, step=50),
                "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "max_depth": t.suggest_int("max_depth", 2, 6),
                "min_child_weight": t.suggest_float("min_child_weight", 1, 10),
                "subsample": t.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10, log=True),
                "reg_alpha": t.suggest_float("reg_alpha", 1e-3, 10, log=True)}
    if name == "LGBM":
        return {"n_estimators": t.suggest_int("n_estimators", 100, 1000, step=50),
                "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "num_leaves": t.suggest_int("num_leaves", 4, 31),
                "min_child_samples": t.suggest_int("min_child_samples", 3, 20),
                "subsample": t.suggest_float("subsample", 0.5, 1.0), "subsample_freq": 1,
                "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10, log=True)}
    if name == "CatB":
        return {"iterations": t.suggest_int("iterations", 300, 1500, step=100),
                "depth": t.suggest_int("depth", 2, 6),
                "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "l2_leaf_reg": t.suggest_float("l2_leaf_reg", 1, 10, log=True)}
    if name == "SVR":
        return {"C": t.suggest_float("C", 0.1, 1000, log=True),
                "epsilon": t.suggest_float("epsilon", 1e-3, 0.3, log=True),
                "gamma": t.suggest_float("gamma", 1e-3, 10, log=True)}
    if name == "ANN":
        n = t.suggest_int("n_layers", 1, 3)
        p = {"n_layers": n, "act": t.suggest_categorical("act", ["relu", "tanh"]),
             "alpha": t.suggest_float("alpha", 1e-5, 1.0, log=True)}
        for i in range(n):
            p[f"n{i}"] = t.suggest_int(f"n{i}", 4, 64, log=True)
        return p
    return {}


MODELS = ["Mean", "MVR", "Ridge", "RF", "ET", "XGB", "LGBM", "CatB", "SVR", "GPR", "ANN"]
print(f"{len(MODELS)} models registered:", MODELS)

# ## 2.4 Nested hyper-parameter tuning (Optuna, inner grouped 5-fold CV)
# `N_TRIALS=40` and `n_repeats=10` reproduce the paper's reported numbers;
# reduce both for a fast debug pass (e.g. 10 / 2) before committing the
# full run.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 40  # reduce to ~10 for a quick debug pass


def tune(name, Xtr, ytr, gtr):
    if name in ("Mean", "MVR", "GPR"):
        return {}
    cv = GroupKFold(n_splits=5)

    def obj(t):
        est = build(name, space(name, t))
        return -cross_val_score(est, Xtr, ytr, groups=gtr, cv=cv,
                                 scoring="neg_root_mean_squared_error").mean()

    st = optuna.create_study(direction="minimize",
                              sampler=optuna.samplers.TPESampler(seed=SEED))
    st.optimize(obj, n_trials=N_TRIALS, show_progress_bar=False)
    return space(name, optuna.trial.FixedTrial(st.best_params))

# ## 2.5 Benchmark runner — Phases 2–4 combined
# For every outer fold of a protocol: tune each model on the training fold
# only, fit, predict on the held-out fold, log metrics + out-of-fold (OOF)
# predictions + the winning hyper-parameters.

In [ ]:
def run_protocol(proto_name, splits, models=MODELS):
    rows, oof, params = [], [], []
    for rep, fold, tr, te in splits:
        for name in models:
            t0 = time.time()
            p = tune(name, X[tr], y[tr], DUP_GROUP[tr])
            est = build(name, p).fit(X[tr], y[tr])
            pred = est.predict(X[te])
            m = metrics(y[te], pred) if len(te) > 1 else {}
            rows.append({"protocol": proto_name, "rep": rep, "fold": fold, "model": name,
                         "n_test": len(te), "sec": round(time.time() - t0, 2), **m})
            oof += [{"protocol": proto_name, "rep": rep, "model": name, "idx": i,
                     "y": yt, "pred": yp} for i, yt, yp in zip(te, y[te], pred)]
            params.append({"protocol": proto_name, "rep": rep, "fold": fold,
                           "model": name, "params": p})
        print(f"[{proto_name}] rep={rep} fold={fold} done ({len(models)} models)")
    res, oof = pd.DataFrame(rows), pd.DataFrame(oof)
    res.to_csv(OUT / f"tables/folds_{proto_name}.csv", index=False)
    oof.to_csv(OUT / f"tables/oof_{proto_name}.csv", index=False)
    json.dump(params, open(OUT / f"tables/params_{proto_name}.json", "w"), default=str)
    return res, oof


res_P1, oof_P1 = run_protocol("P1_paper", paper_split())
res_P2, oof_P2 = run_protocol("P2_rgkf", repeated_group_kfold(DUP_GROUP, 5, 10))
res_P3, oof_P3 = run_protocol("P3_loso", loso_splits(SITE))

# ## 2.6 Summary tables (Table 4 of the paper)
# For P3 (LOSO) we pool out-of-fold predictions per repeat before scoring,
# because several held-out sites have too few blasts for a stable per-fold
# R².

In [ ]:
def summarise(res, oof, pooled=False):
    if pooled:
        return (oof.groupby(["model", "rep"]).apply(lambda d: pd.Series(metrics(d.y, d.pred)))
                .groupby("model").mean().sort_values("RMSE"))
    agg = res.groupby("model")[["R2", "RMSE", "MAE", "MAPE", "VAF", "a20"]].agg(["mean", "std"])
    return agg.sort_values(("RMSE", "mean"))


T4 = {"P1": summarise(res_P1, oof_P1), "P2": summarise(res_P2, oof_P2),
      "P3_pooled": summarise(res_P3, oof_P3, pooled=True)}
for name, table in T4.items():
    table.round(4).to_csv(OUT / f"tables/T4_results_{name}.csv")
    print(f"\n=== {name} ===")
    display(table.round(3))

# ## 2.7 Comparison with literature on the 12 shared validation blasts
# (Table 7 of the paper)

In [ ]:
o = oof_P1.merge(df[["blast_id"]], left_on="idx", right_index=True)
comp = {"Kuznetsov (Kulatilake 2010)": metrics(k.X50, k.blast_id.map(KUZ)),
        "MVR (Kulatilake 2010)": metrics(k.X50, k.blast_id.map(MVR_PUB)),
        "ANN (Kulatilake 2010)": metrics(k.X50, k.blast_id.map(ANN_PUB))}
for mname, d in o[o.blast_id.isin(KUZ)].groupby("model"):
    comp[f"{mname} (this study)"] = metrics(d.y, d.pred)
T7 = pd.DataFrame(comp).T.sort_values("RMSE")
T7.round(3).to_csv(OUT / "tables/T7_vs_literature.csv")
T7.round(3)

# ## 2.8 Statistical significance — Phase 5
# Corrected resampled t-test (Nadeau & Bengio, 2003) between the best model
# (lowest mean RMSE under P2) and every other model, computed on the 50
# outer-fold RMSE values, with Holm correction for multiple comparisons,
# plus an omnibus Friedman test across all models (Table 5 of the paper).

In [ ]:
from statsmodels.stats.multitest import multipletests


def corrected_ttest(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b)
    J = len(d)
    var = np.var(d, ddof=1) * (1 / J + n_test / n_train)
    t = d.mean() / np.sqrt(var)
    return t, 2 * stats.t.sf(abs(t), J - 1)


piv = res_P2.pivot_table(index=["rep", "fold"], columns="model", values="RMSE")
best_model = piv.mean().idxmin()
n_te = res_P2.n_test.mean()
n_tr = len(df) - n_te
tests = [(m, *corrected_ttest(piv[m], piv[best_model], n_tr, n_te)) for m in piv if m != best_model]
T5 = pd.DataFrame(tests, columns=["model", "t", "p"])
T5["p_holm"] = multipletests(T5.p, method="holm")[1]
T5.to_csv(OUT / "tables/T5_significance.csv", index=False)
print("Best model (mean RMSE, P2):", best_model)
display(T5.round(4))
print("Friedman test across all models:", stats.friedmanchisquare(*[piv[m] for m in piv]))

# ## 2.9 Figures — CV box plot and predicted-vs-measured (Figures 5–6)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(11, 4.5))
order = piv.mean().sort_values().index
sns.boxplot(data=piv[order], ax=ax)
ax.set_ylabel("RMSE (m), 10×5 grouped CV")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUT / "figures/F5_cv_boxplot.png", dpi=300)
plt.show()

top_models = list(order[:3]) + ["MVR"]
fig, axes = plt.subplots(1, len(top_models), figsize=(4.2 * len(top_models), 4.2))
for ax, m in zip(axes, top_models):
    d = oof_P2[(oof_P2.model == m) & (oof_P2.rep == 0)]
    mm = metrics(d.y, d.pred)
    lim = [0, 1.0]
    ax.scatter(d.y, d.pred, s=18)
    ax.plot(lim, lim, "k-")
    ax.plot(lim, [v * 1.2 for v in lim], "k:")
    ax.plot(lim, [v * .8 for v in lim], "k:")
    ax.set_title(f"{m}  R2={mm['R2']:.2f}  RMSE={mm['RMSE']:.3f}")
    ax.set_xlabel("Measured X50 (m)")
    ax.set_ylabel("Predicted X50 (m)")
plt.tight_layout()
plt.savefig(OUT / "figures/F6_pred_vs_meas.png", dpi=300)
plt.show()

# ## Outputs of this notebook (→ save as a Kaggle Dataset for Notebooks 03–04)
# - `tables/folds_*.csv`, `oof_*.csv`, `params_*.json` (raw fold-level results)
# - `tables/T4_results_*.csv`, `T5_significance.csv`, `T7_vs_literature.csv`
# - `figures/F5_cv_boxplot.png`, `F6_pred_vs_meas.png`
#
# **Next:** click **"Save Version" → New Dataset from output** (or manually
# create a Kaggle Dataset from `/kaggle/working`), e.g. slug
# `rock-blast-benchmark-outputs`. Notebooks 03 and 04 load it as a second
# Kaggle input alongside `blast_master_110.csv`.